# 10 — Enhancement Round 4 (subtraction: leave-one-group-out)

Instead of adding features, Round 4 removes each engineered group from Final to find dead weight. Rule: SIMPLIFY iff random loss <= 0.001 AND group gain >= +0.002. No test labels, downloads, or leaderboard use.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
A = json.loads((ROOT/'reports/gap5_r4a.json').read_text())
B = json.loads((ROOT/'reports/gap5_r4b.json').read_text())
RF = R['best_cv']
assert abs(A['results']['base_random']['mean'] - RF) < 1e-9
assert abs(B['base_recomputed']['random']['mean'] - RF) < 1e-9
print('bases exact')
print('R4A verdicts:', A['verdicts'])
print('R4B verdicts:', {k: v['decision'] for k, v in B['variants'].items()})
m = B['variants']['A_minus_TicketGroupSize']
print('ticket-drop: random %.4f group %.4f' % (m['random']['mean'], m['group']['mean']))
f = B['variants']['B_minus_FarePerPerson']
print('farepp-drop: random %.4f group %.4f (5/5 group wins, random cost)' % (f['random']['mean'], f['group']['mean']))
assert A['verdicts']['c_minus_mother'] == 'SIMPLIFY' and m['decision'] == 'SIMPLIFY'
print('R4 gate outcomes reproduce from files')

## Round-4 outcome + adjudication: production FROZEN despite two SIMPLIFY verdicts

Mother-drop (random bit-identical, group +0.0022) and TicketGroupSize-drop (random +0.0022, group +0.0090) pass the narrow R4 gates. They are NOT applied, for stated reasons:

- The R4 gate (+0.002 group, no t-requirement) is ~1/6 of the program's uniform noise standard (seed range 0.0134; adoption bars set at/beyond noise). R3C-R2 gained +0.0169 with t=1.98 and was rejected — accepting +0.0022 on a weaker gate would be incoherent. The R4 gate was miscalibrated; the workers executed it correctly.
- Any production change breaks the verified pipeline-to-submission-to-score chain and forces a re-upload for noise-level differences.
- Removing TicketGroupSize while keeping FarePerPerson (computed from those counts) is representationally incoherent.

Recorded as E22–E23. Both findings marked for any future rebuild; production and submission unchanged.
